<a href="https://colab.research.google.com/github/iav2002/AppliedDeepLearning/blob/main/part2_DATAPREP_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applied Deep Learning, Part 2: Dataset Preparation
## 1. Setup

Produces the canonical splits used by every other notebook. Outputs five CSVs into `data_splits/`.

1. test, 10% held out, never touched until final eval
2. train and val, 75% and 15% of the remainder, stratified
3. block 1 and block 2, 50/50 of non-test data, used in Part 3 for autoencoder vs classifier. Might revisit later

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# rebuild the labels df, same logic as notebook 1
# doing it here so notebook 2 is self-contained
DATASET_ROOT = Path("face_age")

rows = []
for age_dir in sorted(DATASET_ROOT.iterdir()):
    if not age_dir.is_dir():
        continue
    age = int(age_dir.name)
    for f in age_dir.iterdir():
        if f.is_file():
            rows.append({"path": str(f), "age": age, "fname": f.name})

df = pd.DataFrame(rows)
print("loaded", len(df), "rows")

loaded 9778 rows


## 2. Categories and sparse age filter

Apply the 7 bins decided in EDA. Drop ages with fewer than 5 images (decided in EDA), this affects ages 91, 95, 99, 100, 101, 110 (those are the ones the EDA flagged).

In [ ]:
# bins from EDA, 7 categories
bins = [0, 5, 13, 20, 31, 46, 61, 200]
labels = ["infant", "child", "teen", "youth", "mid", "mature", "senior"]
df["age_category"] = pd.cut(df["age"], bins=bins, labels=labels, right=False)

# count images per age, then keep only ages with at least 5 samples
age_counts = df["age"].value_counts()
keep_ages = age_counts[age_counts >= 5].index
before = len(df)
df = df[df["age"].isin(keep_ages)].reset_index(drop=True)
after = len(df)
print(f"dropped {before - after} rows from sparse ages")
print(f"remaining {after} rows across {df['age'].nunique()} ages")
print("\ncategory counts after filter")
print(df["age_category"].value_counts().reindex(labels))

dropped 18 rows from sparse ages
remaining 9760 rows across 91 ages

category counts after filter
age_category
infant    2131
child     1124
teen       909
youth     1626
mid       1306
mature    1369
senior    1295
Name: count, dtype: int64


## 3. Test split

Hold out 10% of the data as test. Stratified by category so every class is proportionally represented. This set is sacred, never used for validation, only touched at final eval.

In [ ]:
# stratify by age_category so the test set mirrors the overall category distribution
# random_state fixed so anyone re-running this gets the exact same split
trainval_df, test_df = train_test_split(
    df,
    test_size=0.10,
    stratify=df["age_category"],
    random_state=42,
)

print(f"trainval {len(trainval_df)}    test {len(test_df)}")
print("\ntest category distribution")
print(test_df["age_category"].value_counts().reindex(labels))

trainval 8784    test 976

test category distribution
age_category
infant    213
child     112
teen       91
youth     163
mid       131
mature    137
senior    129
Name: count, dtype: int64


## 4. Train and validation split

From the 90% trainval, split off 15% as validation. Stratified by category again. Final ratios over the full dataset, train 75%, val 15%, test 10%.

Note that 15% of trainval is not the same as 15% of the full dataset. We compute the right fraction explicitly so the final split is 75/15/10 of the original.

In [ ]:
# we want 15% of the original dataset as val
# we have 90% of the original in trainval, so val should be 15/90 of trainval
val_fraction_of_trainval = 0.15 / 0.90

train_df, val_df = train_test_split(
    trainval_df,
    test_size=val_fraction_of_trainval,
    stratify=trainval_df["age_category"],
    random_state=42,
)

print(f"train {len(train_df)}    val {len(val_df)}    test {len(test_df)}")
print(f"ratios   {len(train_df)/len(df):.2%} / {len(val_df)/len(df):.2%} / {len(test_df)/len(df):.2%}")

train 7320    val 1464    test 976
ratios   75.00% / 15.00% / 10.00%


## 5. Block 1 and Block 2

For Part 3,  we haev to split the non-test data into two disjoint blocks. Block 1 trains the autoencoder, Block 2 trains the classifier that uses the AE encoder. Disjoint blocks prevent any leakage between the pretraining and the downstream task.

Block 1 and Block 2 are independent of train/val/test. They overlap with train and val (both come from trainval), but neither contains anything from test. We use 50/50 for now, may revisit when we get to Part 3.

In [ ]:
# split the trainval pool into two halves
# arranged by category so each block is representative
block1_df, block2_df = train_test_split(
    trainval_df,
    test_size=0.50,
    stratify=trainval_df["age_category"],
    random_state=42,
)

print(f"block1 {len(block1_df)}    block2 {len(block2_df)}")
print("\nblock1 category distribution")
print(block1_df["age_category"].value_counts().reindex(labels))

block1 4392    block2 4392

block1 category distribution
age_category
infant    959
child     506
teen      409
youth     732
mid       587
mature    616
senior    583
Name: count, dtype: int64


## 6. Save CSVs and verify

Write all five splits to `../data_splits/`. Then verify no row is in more than one split it should not be in, and confirm the category distributions are preserved.

In [ ]:
# write each split as csv, tracked in git so anyone can reproduce
SPLITS_DIR = Path("data_splits")
SPLITS_DIR.mkdir(exist_ok=True)

splits = {
    "train":  train_df,
    "val":    val_df,
    "test":   test_df,
    "block1": block1_df,
    "block2": block2_df,
}

for name, sub in splits.items():
    out = SPLITS_DIR / f"{name}.csv"
    sub.to_csv(out, index=False)
    print(f"saved {out}  ({len(sub)} rows)")

saved data_splits/train.csv  (7320 rows)
saved data_splits/val.csv  (1464 rows)
saved data_splits/test.csv  (976 rows)
saved data_splits/block1.csv  (4392 rows)
saved data_splits/block2.csv  (4392 rows)


In [ ]:
# sanity 1, test must not overlap with train or val
test_paths = set(test_df["path"])
train_paths = set(train_df["path"])
val_paths = set(val_df["path"])

assert test_paths.isdisjoint(train_paths), "test leaked into train"
assert test_paths.isdisjoint(val_paths),   "test leaked into val"

# sanity 2, block1 and block2 must be disjoint
block1_paths = set(block1_df["path"])
block2_paths = set(block2_df["path"])
assert block1_paths.isdisjoint(block2_paths), "block1 and block2 overlap"

# sanity 3, block1+block2 must equal trainval, they together must not touch test
assert (block1_paths | block2_paths).isdisjoint(test_paths), "blocks leak into test"

print("all sanity checks passed")

all sanity checks passed


In [ ]:
# verify class proportions are preserved across all splits
# small drift is fine, big drift means stratification broke
print("category distribution as fraction of each split\n")
for name, sub in splits.items():
    fracs = sub["age_category"].value_counts(normalize=True).reindex(labels)
    print(f"{name:8s}  " + "  ".join(f"{c}={f:.2f}" for c, f in fracs.items()))

category distribution as fraction of each split

train     infant=0.22  child=0.12  teen=0.09  youth=0.17  mid=0.13  mature=0.14  senior=0.13
val       infant=0.22  child=0.12  teen=0.09  youth=0.17  mid=0.13  mature=0.14  senior=0.13
test      infant=0.22  child=0.11  teen=0.09  youth=0.17  mid=0.13  mature=0.14  senior=0.13
block1    infant=0.22  child=0.12  teen=0.09  youth=0.17  mid=0.13  mature=0.14  senior=0.13
block2    infant=0.22  child=0.12  teen=0.09  youth=0.17  mid=0.13  mature=0.14  senior=0.13
